# PREVOĐENJE PROGRAMSKIH JEZIKA - 4. laboratorijska vježba

Svrha ove bilježnice je provesti vas kroz jednu od mogućnosti simulacije generiranih kodova i provjere njihove ispravnosti.

## Upute za postavljanje okruženja

Za ispravno izvođenje ove bilježnice potrebno je pratiti sljedeće korake:

1. **Preuzimanje alata**: Preuzmite arhivu sa stranice ARM GNU Toolchain. Korisnici Windows sustava trebaju preuzeti: arm-gnu-toolchain-15.2.rel1-mingw-w64-x86_64-arm-none-eabi.zip.

2. **Konfiguracija direktorija**: Raspakirajte arhivu i mapu bin postavite izravno u direktorij u kojem se nalazi ova bilježnica.

3. **Instalacija biblioteka**: Potrebno je instalirati Python biblioteku unicorn. To možete učiniti pokretanjem ćelije koja se nalazi neposredno ispod ovih uputa.

4. **Priprema testova**: Preuzmite arhivu s testovima sa službene stranice predmeta. Sve testne mape kopirajte u direktorij tests unutar vašeg radnog direktorija.

**Očekivana struktura direktorija**

Prije izvršavanja bilježnice, osigurajte da vaš radni direktorij izgleda ovako:

<pre>

radni_direktorij/
├── bin/                    # Mapa s ARM alatima
├── GeneratorKoda           # Izvorni kod vašeg generatora
├── tests/                  # Mapa s testnim primjerima
│   ├── 01_ret_broj/        # Pojedinačni testni slučaj
│   │   ├── test.in         # Ulazni podaci
│   │   └── test.out        # Očekivani izlaz
│   └── ...                 # Ostali testovi
└── simulator.ipynb         # Ova bilježnica

</pre>

In [ ]:
%pip install unicorn

## Simulacija

In [ ]:
import os
import subprocess
import tempfile

import unicorn.arm_const as arm
from unicorn import Uc, UC_ARCH_ARM, UC_MODE_ARM, UC_HOOK_INTR

ARM_CODE_ADDRESS = 0x10000
ARM_CODE_SIZE = 0x4000
MEMORY_SIZE = 0x10000

arm_toolchain_path = ".\\bin"

if arm_toolchain_path not in os.environ["PATH"]:
    os.environ["PATH"] = arm_toolchain_path + os.pathsep + os.environ["PATH"]

def hook_intr(mu, intno, user_data):
    if intno == 2:
        mu.emu_stop()


def evaluate_r6(arm_code_path, expected_r6_value):
    with tempfile.TemporaryDirectory() as tmpdir:
        asm_file = os.path.join(tmpdir, "code.s")
        obj_file = os.path.join(tmpdir, "code.o")
        elf_file = os.path.join(tmpdir, "code.elf")
        bin_file = os.path.join(tmpdir, "code.bin")

        with open(arm_code_path, "r") as src, open(asm_file, "w") as dst:
            dst.write(src.read())

        try:
            subprocess.run(
                ["arm-none-eabi-as", asm_file, "-o", obj_file], capture_output=True, check=True, text=True
            )

            subprocess.run([
                "arm-none-eabi-ld",
                "-Ttext", hex(ARM_CODE_ADDRESS),
                obj_file, "-o", elf_file
            ], capture_output=True, check=True, text=True)

            subprocess.run(["arm-none-eabi-objcopy", "-O", "binary", elf_file, bin_file], capture_output=True, check=True, text=True)
        except subprocess.CalledProcessError as e:
            print("Postoji greška u generiranom kodu:")
            print(e.stderr)
            raise e

        with open(bin_file, "rb") as f:
            code = f.read()

    mu = Uc(UC_ARCH_ARM, UC_MODE_ARM)
    mu.hook_add(UC_HOOK_INTR, hook_intr)

    mu.mem_map(0x0000, 0x30000)

    mu.mem_write(ARM_CODE_ADDRESS, code)

    mu.reg_write(arm.UC_ARM_REG_SP, 0x20000)

    try:
        mu.emu_start(ARM_CODE_ADDRESS, ARM_CODE_ADDRESS + len(code))
    except Exception as e:
        pass

    r6_value = mu.reg_read(arm.UC_ARM_REG_R6)

    if r6_value & 0x80000000:
        r6_signed = r6_value - 0x100000000
    else:
        r6_signed = r6_value

    print(f"  > R6 result: {hex(r6_value)} (Signed: {r6_signed}) (Expected: {expected_r6_value})")

    return r6_signed == expected_r6_value


TESTS_FOLDER = "tests"


def run_all_tests():
    test_dirs = sorted([f for f in os.listdir(TESTS_FOLDER) if os.path.isdir(os.path.join(TESTS_FOLDER, f))])
    total = len(test_dirs)
    passed = 0
    for test_dir in test_dirs:
        test_path = os.path.join(TESTS_FOLDER, test_dir)
        print(f"Running {test_path}...")

        input_path = os.path.join(test_path, "test.in")
        output_path = os.path.join(test_path, "test.out")

        # TODO otkomentirajte subprocess koji se odnosi na jezik u kojem ste pisali Vašu laboratorijsku vježbu, ostale zakomentirajte    
        try:
            # python
            # result = subprocess.run(
            #     f"python3 GeneratorKoda.py < {input_path}",
            #     shell=True, check=True, capture_output=True, text=true
            # )

            # java
            subprocess.run(f"javac GeneratorKoda.java", shell=True, check=True, capture_output=True, text=True)
            result = subprocess.run(
                f"java GeneratorKoda < {input_path}",
                shell=True, check=True,  capture_output=True
            )
            # C++
            # subprocess.run(f"g++ GeneratorKoda.cpp -o generator", shell=True, check=True, text=True)
            # result = subprocess.run(
            #     f"./generator < {input_path}",
            #     shell=True, check=True,  capture_output=True, text=True
            # )
        except subprocess.CalledProcessError as e:
            print("Postoji u prevođenju koda:")
            print(e.stderr)
            raise e

        with open(output_path, "r") as f:
            line = f.read().strip()
            expected_val = int(line.strip(), 10)

        if evaluate_r6("a.s", expected_val):
            print("  [PASS]")
            passed += 1
        else:
            print("  [FAIL]")

            with open(os.path.join(".", "a.s"), "r") as f1:
                with open(os.path.join(".", f"{test_dir}.s"), "w") as f2:
                    f2.write(f1.read())

    print("-" * 20)
    print(f"FINAL RESULT: {passed}/{total} tests passed.")


if __name__ == "__main__":
    run_all_tests()
